# Calculate interannotator agreement


## Setup

- Enable `%autoreload` for immediate reflection of utils library changes
- Load credentials from the `.env` file
- Configure the environment and establish connection to the Argilla server

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import argilla as rg
import matplotlib.pyplot as plt
import seaborn as sns
from archaeo_ner_greek.utils import get_argilla_client, load_credentials_from_env, get_dataset_as_dataframe
from archaeo_ner_greek.utils import prepare_iaa_data, calculate_ner_iaa, calculate_label_iaa, analyze_iaa_discrepancies
from archaeo_ner_greek.logging_config import setup_logging
import logging

# 1. Setup logging
setup_logging()

# 2. Load configuration from .env
env_values = load_credentials_from_env()

# 3. Establish connection to Argilla
if env_values:
    client = get_argilla_client(env_values)
else:
    logging.warning("Missing configuration in .env file.")

## Loading Annotations from Argilla

- Fetch the `archaeo_ner_greek` dataset
- Convert records into a Pandas DataFrame using `utils.py`
- Ensure all user responses (spans and suggestions) are included for analysis

In [ ]:
# Define dataset and workspace
workspace_name = "atrium"
dataset_name = "atrium_csd_ner_annotations"
      
# Fetch the full dataframe with responses included
df = get_dataset_as_dataframe(
    client, 
    dataset_name=dataset_name, 
    workspace_name=workspace_name, 
    include_responses=True
)

## Data Exploration & Quality Check

- Visualize raw data before metric calculation
- Use custom CSS styling for wrapping Greek text and response dictionaries
- Manually verify annotation density and structure

In [ ]:
# 1. Set display option to show full content instead of truncating (...)
pd.set_option('display.max_colwidth', None)

# 2. Select columns and apply styling for wrapping and vertical alignment
(df[["sentence_field", "responses"]].head(10).style
 .set_properties(**{
     'text-align': 'left',
     'white-space': 'pre-wrap',
     'vertical-align': 'top'
 })
 .set_table_styles([
     {'selector': 'th', 'props': [('font-weight', 'bold')]},
     {'selector': 'td.col0', 'props': [('width', '400px')]}, # Fixed width for sentence
     {'selector': 'td.col1', 'props': [('width', '600px')]}  # Fixed width for responses
 ])
)

## Structuring Data for IAA

- Transform raw Argilla responses into entity triplets `(start, end, label)`
- Group annotations by user using the `prepare_iaa_data` utility
- Identify sentences ready for comparison (2+ annotators) versus those requiring more work

In [ ]:
# 1. Prepare data for agreement analysis
iaa_results = prepare_iaa_data(df)
iaa_ready = iaa_results["ready"]

# 2. Map annotators for consistent reporting (A=stalexan, B=sasi.dimopoulou)
annotator_a = env_values.get("ANNOTATOR_A")
annotator_b = env_values.get("ANNOTATOR_B")
print(f"Mapping: A = {annotator_a}, B = {annotator_b}")

# 3. Report annotation status
print(f"✅ Sentences ready for IAA: {len(iaa_ready)}")
print(f"⚠️  Missing teammates (only 1 annotator): {len(iaa_results['missing_teammate'])}")
print(f"❌ Unannotated sentences: {len(iaa_results['unannotated'])}")

## Pairwise Agreement Report (Global)

- Calculate high-level consistency metrics between annotator pairs
- **Sentence Match %**: Perfect sentence set-equality (including sentences with zero entities)
- **Span F1-Score**: Standard NER benchmark for entity segment overlap

In [ ]:
# Calculate and display pairwise agreement report
iaa_report = calculate_ner_iaa(iaa_ready)
iaa_report

## Label-Specific Agreement Breakdown

- Breakdown agreement metrics (F1-score) for each individual label
- Identify categories where annotation guidelines may require clarification or additional examples

In [ ]:
# Calculate and display per-label agreement statistics using consistent A/B mapping
label_report = calculate_label_iaa(iaa_ready, annotators=[annotator_a, annotator_b])
label_report

## Discrepancy Analysis

- Distinguish between classification errors (same span, different label) and boundary errors (overlapping spans)
- Quantify total misses where an entity was identified by only one annotator

In [ ]:
# Categorize disagreements into specific error types
discrepancy_report = analyze_iaa_discrepancies(iaa_ready, annotators=[annotator_a, annotator_b])
discrepancy_report

## Visualizations for Annotator Feedback

- Generate graphical representations of agreement and disagreement
- Compare performance across different entity categories
- Visualize the distribution of specific error types to inform guideline refinement

In [ ]:
# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'DejaVu Sans'

fig, axes = plt.subplots(3, 1, figsize=(10, 18))

# 1. Label F1-Scores
label_data_clean = label_report[label_report["Label"] != "OVERALL"].sort_values("F1-Score", ascending=False)
sns.barplot(data=label_data_clean, x="F1-Score", y="Label", ax=axes[0], palette="viridis")
axes[0].set_title("IAA F1-Score per Label", fontsize=14, fontweight='bold')
axes[0].set_xlim(0, 1)

# 2. Agreement vs Disagreement Volume
label_counts = label_data_clean.melt(id_vars="Label", value_vars=["Both", f"Only in A ({annotator_a})", f"Only in B ({annotator_b})"], 
                                    var_name="Category", value_name="Count")
sns.barplot(data=label_counts, x="Count", y="Label", hue="Category", ax=axes[1], palette="muted")
axes[1].set_title("Annotation Volume Breakdown (Agreement vs Omissions)", fontsize=14, fontweight='bold')
axes[1].legend(title="Source", bbox_to_anchor=(1.05, 1), loc='upper left')

# 3. Discrepancy Types Distribution
discrepancy_data = discrepancy_report.reset_index()
discrepancy_data.columns = ["Error Type", "Count"]
sns.barplot(data=discrepancy_data, x="Count", y="Error Type", ax=axes[2], palette="rocket")
axes[2].set_title("Distribution of Discrepancy Types", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
iaa_report

In [ ]:
label_report

In [ ]:
discrepancy_report